# Assignment 1: Myanmar POS Tagger using CRF (myPOS v3.0)

Builds a Part-of-Speech tagger for Myanmar using the
[myPOS corpus (version 3.0)](https://github.com/ye-kyaw-thu/myPOS/tree/master/corpus-ver-3.0)
and the same CRF approach (`crfsuite`) used in the word-segmentation tutorial
(`CRF-tutorial.ipynb`). This notebook reuses the already-built `crfsuite` tool
and the `crfutils.py` feature-extraction library from that tutorial — only the
data-prep script and feature template are new, since the task (POS tagging)
and the data format (`word/TAG` instead of syllable boundary tags) are
different.

**Data:** the corpus author's own official train/test split —
`train.mypos-ver3.txt` (42,196 sentences) and `otest.1k.txt` (1,000 sentences,
held out).

## Download the tagged myPOS v3.0 corpus

In [1]:
# Create a folder for the POS-tagging assignment data (kept separate from the word-seg tutorial's data)
!mkdir -p /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag

In [2]:
# Download the official train/test split from the myPOS repo (corpus-ver-3.0/corpus)
!curl -sL "https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/master/corpus-ver-3.0/corpus/train.mypos-ver3.txt" -o /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.mypos.txt
!curl -sL "https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/master/corpus-ver-3.0/corpus/otest.1k.txt" -o /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/test.mypos.txt
!wc -l /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.mypos.txt /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/test.mypos.txt

   42196 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.mypos.txt
    1000 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/test.mypos.txt
   43196 total


In [3]:
# Preview the raw myPOS format: space-separated "word/TAG" tokens per sentence;
# compound words are pipe-joined "subword1/TAG1|subword2/TAG2"
!head -n 5 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.mypos.txt

၁၉၆၂/num ခုနှစ်/n ခန့်မှန်း/v သန်းခေါင်စာရင်း/n အရ/ppm လူဦးရေ/n ၁၁၅၉၃၁/num ယောက်/part ရှိ/v သည်/ppm ။/punc
လူ/n တိုင်း/part တွင်/ppm သင့်မြတ်/v လျော်ကန်/v စွာ/part ကန့်သတ်/v ထား/part သည့်/part အလုပ်/n လုပ်/v ချိန်/n အပြင်/conj ၊/punc လစာ/n နှင့်တကွ/conj အခါ/n ကာလ/n အားလျော်စွာ/ppm သတ်မှတ်/v ထား/part သည့်/part အလုပ်/n|အားလပ်ရက်/n များ/part ပါဝင်/v သည့်/part အနားယူခွင့်/n နှင့်/conj အားလပ်ခွင့်/n ခံစားပိုင်ခွင့်/n ရှိ/v သည်/ppm ။/punc
ဤ/adj နည်း/n ကို/ppm စစ်ယူ/v သော/part နည်း/n ဟု/part ခေါ်/v သည်/ppm ။/punc
စာပြန်ပွဲ/n ဆို/v တာ/part က/ppm အာဂုံဆောင်/v အလွတ်ကျက်/v ထား/part တဲ့/part ပိဋကတ်သုံးပုံ/n|စာပေ/n တွေ/part ကို/ppm စာစစ်/v|သံဃာတော်ကြီး/n တွေ/part ရဲ့/ppm ရှေ့/n မှာ/ppm အလွတ်/adv ပြန်/v ပြီး/part ရွတ်ပြ/v ရ/part တာ/part ပေါ့/part ။/punc
ဒီ/pron မှာ/ppm ကျွန်တော့်/pron သက်သေခံကတ်/n ပါ/part ။/punc


## Convert myPOS format into word/tag sequences

`crfutils.py` (from the word-seg tutorial) expects one `word<TAB>tag` per
line, blank line between sentences. myPOS's raw format is space-separated
`word/TAG` tokens, with compound words pipe-joined as
`word1/TAG1|word2/TAG2`. The cell below converts one format into the other,
directly in the notebook (no external script needed).

In [4]:
def mypos_to_wordtag(input_path, output_path):
    """Convert myPOS 'word/TAG' corpus into word<TAB>tag sequences
    (blank line between sentences), the format crfutils.py expects."""
    with open(input_path, 'r') as fi, open(output_path, 'w') as fo:
        for line in fi:
            line = line.strip()
            if not line:
                continue
            for token in line.split():
                for unit in token.split('|'):
                    if not unit:
                        continue  # corpus has a stray trailing '|' on at least one token
                    word, tag = unit.rsplit('/', 1)
                    fo.write(f"{word}\t{tag}\n")
            fo.write("\n")

DATA = "/Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag"
mypos_to_wordtag(f"{DATA}/train.mypos.txt", f"{DATA}/train.postag.txt")
mypos_to_wordtag(f"{DATA}/test.mypos.txt", f"{DATA}/test.postag.txt")
!wc -l {DATA}/train.postag.txt {DATA}/test.postag.txt

  593245 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.postag.txt
   14468 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/test.postag.txt
  607713 total


In [5]:
# Preview the converted training data
!head -n 15 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.postag.txt

၁၉၆၂	num
ခုနှစ်	n
ခန့်မှန်း	v
သန်းခေါင်စာရင်း	n
အရ	ppm
လူဦးရေ	n
၁၁၅၉၃၁	num
ယောက်	part
ရှိ	v
သည်	ppm
။	punc

လူ	n
တိုင်း	part
တွင်	ppm


## Feature extraction (chunking.py, reused as-is)

The word-segmentation tutorial's `chunking.py` already uses `separator='\t'`,
`fields='w y'`, and a ±2-word context window + adjacent-word bigram
templates — that's exactly what word-level POS tagging needs too (word
identity and its neighbors), so it's reused unchanged rather than
duplicated. `chunking.py` and `crfutils.py` are both untouched by this
notebook.

In [6]:
# View the feature template (same file used by the word-segmentation tutorial)
!cat /Users/yadanar/Desktop/AIE-F-B2/notebooks/tool/crfsuite/example/chunking.py

#!/usr/bin/env python

"""
A feature extractor for chunking.
Copyright 2010,2011 Naoaki Okazaki.
"""

# Separator of field values.
separator = '\t'

# Field names of the input data.
fields = 'w y'

# Attribute templates.
templates = (
    (('w', -2), ),
    (('w', -1), ),
    (('w',  0), ),
    (('w',  1), ),
    (('w',  2), ),
    (('w', -1), ('w',  0)),
    (('w',  0), ('w',  1)),
    )


import crfutils

def feature_extractor(X):
    # Apply attribute templates to obtain features (in fact, attributes)
    crfutils.apply_templates(X, templates)
    if X:
	# Append BOS and EOS features manually
        X[0]['F'].append('__BOS__')     # BOS feature
        X[-1]['F'].append('__EOS__')    # EOS feature

if __name__ == '__main__':
    crfutils.main(feature_extractor, fields=fields, sep=separator)


In [7]:
# Make sure we're in crfsuite's example folder before calling chunking.py
%cd /Users/yadanar/Desktop/AIE-F-B2/notebooks/tool/crfsuite/example

/Users/yadanar/Desktop/AIE-F-B2/notebooks/tool/crfsuite/example


In [8]:
# Convert train and test data into CRFsuite feature format
!cat /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.postag.txt | python ./chunking.py > /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.pos.crfsuite.txt
!cat /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/test.postag.txt | python ./chunking.py > /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/test.pos.crfsuite.txt
!wc -l /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.pos.crfsuite.txt /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/test.pos.crfsuite.txt

  593245 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.pos.crfsuite.txt
   14468 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/test.pos.crfsuite.txt
  607713 total


In [9]:
# Preview the CRFsuite-format training data
!head -n 5 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.pos.crfsuite.txt

num	w[0]=၁၉၆၂	w[1]=ခုနှစ်	w[2]=ခန့်မှန်း	w[0]|w[1]=၁၉၆၂|ခုနှစ်	__BOS__
n	w[-1]=၁၉၆၂	w[0]=ခုနှစ်	w[1]=ခန့်မှန်း	w[2]=သန်းခေါင်စာရင်း	w[-1]|w[0]=၁၉၆၂|ခုနှစ်	w[0]|w[1]=ခုနှစ်|ခန့်မှန်း
v	w[-2]=၁၉၆၂	w[-1]=ခုနှစ်	w[0]=ခန့်မှန်း	w[1]=သန်းခေါင်စာရင်း	w[2]=အရ	w[-1]|w[0]=ခုနှစ်|ခန့်မှန်း	w[0]|w[1]=ခန့်မှန်း|သန်းခေါင်စာရင်း
n	w[-2]=ခုနှစ်	w[-1]=ခန့်မှန်း	w[0]=သန်းခေါင်စာရင်း	w[1]=အရ	w[2]=လူဦးရေ	w[-1]|w[0]=ခန့်မှန်း|သန်းခေါင်စာရင်း	w[0]|w[1]=သန်းခေါင်စာရင်း|အရ
ppm	w[-2]=ခန့်မှန်း	w[-1]=သန်းခေါင်စာရင်း	w[0]=အရ	w[1]=လူဦးရေ	w[2]=၁၁၅၉၃၁	w[-1]|w[0]=သန်းခေါင်စာရင်း|အရ	w[0]|w[1]=အရ|လူဦးရေ


## Train the CRF POS-tagging model

In [10]:
# Train, evaluating against the held-out test set every 2 iterations (-e2)
!time crfsuite learn -e2 -m /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/pos.model /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.pos.crfsuite.txt /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/test.pos.crfsuite.txt

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

Start time of the training: 2026-08-01T04:54:40Z

Reading the data set(s)
[1] /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/train.pos.crfsuite.txt
0....1....2....3....4....5....6....7....8....9....10
Number of instances: 42197
Seconds required: 2.017
[2] /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/test.pos.crfsuite.txt
0....1....2....3....4....5....6....7....8....9....10
Number of instances: 1001
Seconds required: 0.049

Statistics the data set(s)
Number of data sets (groups): 2
Number of instances: 43196
Number of items: 564517
Number of attributes: 452474
Number of labels: 15

Holdout group: 2

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 0
0....1....2....3....4....5....6....7....8....9....10
Number of features: 562703
Seconds required: 0.816

L-BFGS optimization
c1: 0.000000
c2: 1.000000
num_memories: 6
max_iterations: 2147483647

## Evaluate on the held-out test set

In [11]:
# Quiet mode (-qt): print only the per-tag and overall accuracy summary
!crfsuite tag -qt -m /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/pos.model /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/test.pos.crfsuite.txt

Performance by label (#match, #model, #ref) (precision, recall, F1):
    num: (130, 133, 155) (0.9774, 0.8387, 0.9028)
    n: (2925, 3190, 3000) (0.9169, 0.9750, 0.9451)
    v: (1897, 2008, 2010) (0.9447, 0.9438, 0.9443)
    ppm: (2020, 2060, 2060) (0.9806, 0.9806, 0.9806)
    part: (3107, 3193, 3189) (0.9731, 0.9743, 0.9737)
    punc: (1268, 1268, 1270) (1.0000, 0.9984, 0.9992)
    conj: (374, 416, 411) (0.8990, 0.9100, 0.9045)
    adj: (285, 320, 366) (0.8906, 0.7787, 0.8309)
    adv: (200, 221, 262) (0.9050, 0.7634, 0.8282)
    pron: (454, 468, 476) (0.9701, 0.9538, 0.9619)
    tn: (135, 138, 142) (0.9783, 0.9507, 0.9643)
    fw: (20, 21, 87) (0.9524, 0.2299, 0.3704)
    int: (21, 21, 25) (1.0000, 0.8400, 0.9130)
    sb: (3, 3, 3) (1.0000, 1.0000, 1.0000)
    abb: (8, 8, 12) (1.0000, 0.6667, 0.8000)
Macro-average precision, recall, F1: (0.959207, 0.853590, 0.887911)
Item accuracy: 12847 / 13468 (0.9539)
Instance accuracy: 610 / 1000 (0.6100)
Elapsed time: 0.025942 [sec] (38547.5 [in

In [12]:
# Tag test data with -r to inspect reference vs. hypothesis tags side by side
!crfsuite tag -r -m /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/pos.model /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/test.pos.crfsuite.txt > /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/ref_hyp.out.txt
!head -n 30 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/postag/ref_hyp.out.txt

tn	tn
n	n
ppm	ppm
n	n
tn	num
part	part
punc	punc

n	n
ppm	ppm
pron	pron
pron	pron
ppm	ppm
v	v
part	part
ppm	ppm
punc	punc

pron	pron
n	n
v	v
v	v
part	part
punc	punc

n	n
v	v
part	part
v	v
ppm	ppm
